Paso 1 – Extracción de Características Acústicas

En esta etapa se realiza la generación automática de variables numéricas (features) a partir de los segmentos de audio previamente etiquetados.

Objetivo
Transformar cada segmento de 5 segundos en un vector de características acústicas que pueda ser utilizado como entrada para un modelo de aprendizaje supervisado.

Procedimiento
Se cargó el dataset etiquetado (dataset_v1_etiquetado_v2.csv).
Para cada fila:
Se localizó el archivo WAV correspondiente.
Se extrajo el segmento definido por start_time_s y end_time_s.
Se calcularon características acústicas representativas relacionadas con:
Nivel y dinámica
Sonoridad (LUFS)
Pico verdadero (True Peak)
Silencio
Correlación estéreo
Contenido espectral

Features generadas

Para cada segmento se obtuvieron las siguientes variables:
rms_mean
crest_factor_db
true_peak_dbfs
short_term_lufs_mean
loudness_var_proxy_db
silence_ratio
stereo_correlation
spectral_centroid_mean
Estas variables permiten representar cuantitativamente los problemas técnicos previamente etiquetados (distorsión, problemas de fase, silencio anómalo, niveles extremos, etc.).

Validación

Se verificó que:
No existan valores nulos (NaN).
Los rangos de cada variable sean físicamente coherentes.
La distribución del dataset esté balanceada para clasificación binaria (OK vs NO_OK).

Resultado

Se generó el archivo:

features_dataset_v1.csv

El cual contiene:
Identificadores del segmento
Variable objetivo binaria (y_bin)
Conjunto completo de características acústicas
Este archivo será utilizado en el Paso 2 para el entrenamiento del modelo supervisado.

Conclusión

El dataset ya se encuentra listo para iniciar el entrenamiento del modelo de clasificación binaria.

In [1]:
pip install pandas numpy soundfile librosa pyloudnorm

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pyloudnorm

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pyloudnorm as pyln
print("pyloudnorm OK")

pyloudnorm OK


CELDA 1: IMPORTACIONES Y CONFIGURACIONES

In [4]:
import os
import math
import numpy as np
import pandas as pd

# Audio IO
import soundfile as sf

# Features
import librosa

# Loudness (LUFS) - opcional pero recomendado
try:
    import pyloudnorm as pyln
    HAS_PYLOUDNORM = True
except ImportError:
    HAS_PYLOUDNORM = False

BASE_DIR = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0"
CSV_PATH = os.path.join(BASE_DIR, "dataset_v1_etiquetado_v2.csv")
WAV_DIR  = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\WAVS ENTRENAMIENTO"

OUT_FEATURES_CSV = os.path.join(r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1", "features_dataset_v1.csv")

print("pyloudnorm disponible:", HAS_PYLOUDNORM)
print("CSV_PATH:", CSV_PATH)
print("WAV_DIR:", WAV_DIR)

pyloudnorm disponible: True
CSV_PATH: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 0\dataset_v1_etiquetado_v2.csv
WAV_DIR: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\WAVS ENTRENAMIENTO


CELDA 2 - CARGAR DATASET V1 Y VALIDAR COLUMNAS

In [5]:
df = pd.read_csv(CSV_PATH)

required_cols = ["file_name", "start_time_s", "end_time_s", "qc_result", "y_bin"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

# Asegurar tipos numéricos de tiempos
df["start_time_s"] = pd.to_numeric(df["start_time_s"], errors="coerce")
df["end_time_s"]   = pd.to_numeric(df["end_time_s"], errors="coerce")

bad_time = df[df["start_time_s"].isna() | df["end_time_s"].isna()]
if len(bad_time) > 0:
    raise ValueError(f"Hay filas con tiempos inválidos. Ejemplo:\n{bad_time.head()}")

print("Filas:", len(df))
df.head()

Filas: 793


,clip_id,file_name,start_time_s,end_time_s,duration_s,audio_layout,program_type,segment_type,lufs_integrated,true_peak_dbfs,...,source_format,source_origin,pre_correction,qc_result,annotator,synthetic_case,synthetic_type,confidence,notes,y_bin
0,100ECDICEN_PGR276 B1_FULL MIX_1,100ECDICEN_PGR276 B1_FULL MIX.wav,0,5,5,STEREO,SERIE,MIXTO,NaN,NaN,...,WAV,CANAL,0,OK,AA,0,0,5,NaN,0
1,100ECDICEN_PGR276 B1_FULL MIX_2,100ECDICEN_PGR276 B1_FULL MIX.wav,5,10,5,STEREO,SERIE,MIXTO,NaN,NaN,...,WAV,CANAL,0,OK,AA,0,0,5,NaN,0
2,100ECDICEN_PGR276 B1_FULL MIX_3,100ECDICEN_PGR276 B1_FULL MIX.wav,10,15,5,STEREO,SERIE,MIXTO,NaN,NaN,...,WAV,CANAL,0,OK,AA,0,0,5,NaN,0
3,100ECDICEN_PGR276 B1_FULL MIX_4,100ECDICEN_PGR276 B1_FULL MIX.wav,15,20,5,STEREO,SERIE,MIXTO,NaN,NaN,...,WAV,CANAL,0,OK,AA,0,0,5,NaN,0
4,100ECDICEN_PGR276 B1_FULL MIX_5,100ECDICEN_PGR276 B1_FULL MIX.wav,20,25,5,STEREO,SERIE,MIXTO,NaN,NaN,...,WAV,CANAL,0,OK,AA,0,0,5,NaN,0


Celda 3 — Funciones base (cargar segmento + features)

In [ ]:
def load_segment(wav_path: str, start_s: float, end_s: float):
    """Carga un segmento [start_s, end_s) desde wav_path sin escribir archivos."""
    if end_s <= start_s:
        raise ValueError(f"end_s <= start_s: {start_s} - {end_s}")

    info = sf.info(wav_path)
    sr = info.samplerate
    channels = info.channels

    start_frame = int(round(start_s * sr))
    end_frame = int(round(end_s * sr))
    frames = max(0, end_frame - start_frame)

    if start_frame < 0:
        start_frame = 0

    # Leer frames del segmento
    with sf.SoundFile(wav_path, mode="r") as f:
        f.seek(start_frame)
        audio = f.read(frames, dtype="float32", always_2d=True)  # shape: (n, ch)

    # Si viene mono, lo dejamos con shape (n,1)
    if audio.ndim != 2:
        audio = np.atleast_2d(audio).T

    # safety: si el archivo es más corto de lo esperado
    if audio.shape[0] == 0:
        raise ValueError("Segmento vacío (0 muestras).")

    return audio, sr, channels

def stereo_correlation(x: np.ndarray):
    """Correlación L/R (si no es estéreo, devuelve 1.0)."""
    if x.shape[1] < 2:
        return 1.0
    L = x[:, 0]
    R = x[:, 1]
    # Evitar NaN por señales constantes
    if np.std(L) < 1e-8 or np.std(R) < 1e-8:
        return 0.0
    return float(np.corrcoef(L, R)[0, 1])

def silence_ratio_db(x_mono: np.ndarray, thresh_db: float = -60.0):
    """Porcentaje de muestras bajo un umbral en dBFS aproximado (mono)."""
    eps = 1e-12
    mag = np.abs(x_mono) + eps
    db = 20.0 * np.log10(mag)
    return float(np.mean(db < thresh_db))

def rms_mean(x_mono: np.ndarray):
    return float(np.sqrt(np.mean(np.square(x_mono)) + 1e-12))

def true_peak_dbfs(x: np.ndarray):
    """True Peak aproximado: oversampling x4 con librosa.resample (proxy)."""
    # proxy: upsample mono mix a 4x y medir pico
    x_mono = np.mean(x, axis=1)
    # Si el segmento es corto, igual funciona
    sr_dummy = 48000  # no usamos sr real aquí; se ajusta en la llamada
    return None  # se calcula en compute_features para usar sr real

def compute_features(audio: np.ndarray, sr: int):
    """
    audio: (n_samples, n_channels) float32
    Devuelve un dict con features baseline.
    """
    # Mono para varias métricas
    x_mono = np.mean(audio, axis=1)

    absx = np.abs(x_mono)
    clipping_ratio = float(np.mean(absx >= 0.999))
    near_ceiling_ratio = float(np.mean(absx >= 0.98))

    # RMS
    rms = rms_mean(x_mono)

    # Pico sample-peak
    sample_peak = float(np.max(np.abs(x_mono)) + 1e-12)
    sample_peak_db = 20.0 * np.log10(sample_peak)

    # Crest factor (en dB): pico - RMS
    crest = float(sample_peak_db - (20.0 * np.log10(rms + 1e-12)))

    # True Peak proxy: oversampling 4x y medir pico
    try:
        x_up = librosa.resample(x_mono, orig_sr=sr, target_sr=sr*4)
        tp = float(np.max(np.abs(x_up)) + 1e-12)
        true_peak_db = float(20.0 * np.log10(tp))
    except Exception:
        true_peak_db = float(sample_peak_db)  # fallback

    # Loudness short-term (LUFS) si pyloudnorm está disponible
    if HAS_PYLOUDNORM:
        meter = pyln.Meter(sr)  # ITU-R BS.1770
        # Para segmentos cortos, integrated sobre el segmento funciona como "short-term promedio"
        try:
            lufs = float(meter.integrated_loudness(x_mono))
        except Exception:
            lufs = np.nan
    else:
        lufs = np.nan  # luego lo podemos llenar si instalas pyloudnorm

    # Variación de loudness (proxy) con RMS frameado
    try:
        hop = int(0.1 * sr)  # 100 ms
        frame = int(0.4 * sr)  # 400 ms
        rms_frames = librosa.feature.rms(y=x_mono, frame_length=frame, hop_length=hop)[0]
        rms_db_frames = 20.0 * np.log10(rms_frames + 1e-12)
        lufs_std_proxy = float(np.std(rms_db_frames))
    except Exception:
        lufs_std_proxy = np.nan

    # Silencio
    sil_ratio = silence_ratio_db(x_mono, thresh_db=-60.0)

    # Correlación estéreo
    corr = stereo_correlation(audio)

    # Espectral centroid (promedio)
    try:
        centroid = librosa.feature.spectral_centroid(y=x_mono, sr=sr)[0]
        centroid_mean = float(np.mean(centroid))
    except Exception:
        centroid_mean = np.nan

    # Spectral flatness (promedio)
    # Útil para distorsión/saturación armónica (sin clipping duro)
    try:
        flat = librosa.feature.spectral_flatness(y=x_mono)[0]
        spectral_flatness_mean = float(np.mean(flat))
    except Exception:
        spectral_flatness_mean = np.nan


    return {
        "rms_mean": rms,
        "crest_factor_db": crest,
        "true_peak_dbfs": true_peak_db,
        "short_term_lufs_mean": lufs,
        "loudness_var_proxy_db": lufs_std_proxy,
        "silence_ratio": sil_ratio,
        "stereo_correlation": corr,
        "spectral_centroid_mean": centroid_mean,
        "spectral_flatness_mean": spectral_flatness_mean,
        "clipping_ratio": clipping_ratio,
        "near_ceiling_ratio": near_ceiling_ratio,
    }

Celda 4 — Prueba con 1 fila (para verificar que todo está bien)

In [7]:
i = 0  # cambia a cualquier índice
row = df.iloc[i]

wav_path = os.path.join(WAV_DIR, row["file_name"])
print("Probando:", wav_path)

audio, sr, ch = load_segment(wav_path, row["start_time_s"], row["end_time_s"])
feat = compute_features(audio, sr)

print("SR:", sr, "CH:", ch, "samples:", audio.shape[0])
feat

Probando: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\WAVS ENTRENAMIENTO\100ECDICEN_PGR276 B1_FULL MIX.wav
SR: 48000 CH: 2 samples: 240000


{'rms_mean': 0.042636506259441376,
 'crest_factor_db': 16.422409958272738,
 'true_peak_dbfs': -10.98205682459411,
 'short_term_lufs_mean': -27.601156560550525,
 'loudness_var_proxy_db': 1.02773916721344,
 'silence_ratio': 0.019679166666666668,
 'stereo_correlation': 0.698463633564033,
 'spectral_centroid_mean': 2865.7882048081224,
 'spectral_flatness_mean': 0.001436307793483138,
 'clipping_ratio': 0.0,
 'near_ceiling_ratio': 0.0}

Celda 5 — Extracción masiva de features + guardado

In [8]:
records = []
errors = []

for idx, row in df.iterrows():
    wav_path = os.path.join(WAV_DIR, str(row["file_name"]))

    try:
        audio, sr, ch = load_segment(wav_path, float(row["start_time_s"]), float(row["end_time_s"]))
        feat = compute_features(audio, sr)

        rec = {
            "clip_id": row.get("clip_id", f"row_{idx}"),
            "file_name": row["file_name"],
            "start_time_s": row["start_time_s"],
            "end_time_s": row["end_time_s"],
            "qc_result": row["qc_result"],
            "y_bin": row["y_bin"],
        }
        rec.update(feat)
        records.append(rec)

    except Exception as e:
        errors.append({"row": idx, "file_name": row["file_name"], "err": str(e)})

features_df = pd.DataFrame(records)

print("Features extraídas:", len(features_df))
print("Errores:", len(errors))

if len(errors) > 0:
    err_df = pd.DataFrame(errors)
    err_path = os.path.join(r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1", "feature_extraction_errors.csv")
    err_df.to_csv(err_path, index=False, encoding="utf-8")
    print("Errores guardados en:", err_path)

features_df.to_csv(OUT_FEATURES_CSV, index=False, encoding="utf-8")
print("Features guardadas en:", OUT_FEATURES_CSV)

print(features_df.columns.tolist())
features_df.head()

Features extraídas: 764
Errores: 29
Errores guardados en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\feature_extraction_errors.csv
Features guardadas en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\features_dataset_v1.csv
['clip_id', 'file_name', 'start_time_s', 'end_time_s', 'qc_result', 'y_bin', 'rms_mean', 'crest_factor_db', 'true_peak_dbfs', 'short_term_lufs_mean', 'loudness_var_proxy_db', 'silence_ratio', 'stereo_correlation', 'spectral_centroid_mean', 'spectral_flatness_mean', 'clipping_ratio', 'near_ceiling_ratio']


,clip_id,file_name,start_time_s,end_time_s,qc_result,y_bin,rms_mean,crest_factor_db,true_peak_dbfs,short_term_lufs_mean,loudness_var_proxy_db,silence_ratio,stereo_correlation,spectral_centroid_mean,spectral_flatness_mean,clipping_ratio,near_ceiling_ratio
0,100ECDICEN_PGR276 B1_FULL MIX_1,100ECDICEN_PGR276 B1_FULL MIX.wav,0,5,OK,0,0.042637,16.422410,-10.982057,-27.601157,1.027739,0.019679,0.698464,2865.788205,0.001436,0.0,0.0
1,100ECDICEN_PGR276 B1_FULL MIX_2,100ECDICEN_PGR276 B1_FULL MIX.wav,5,10,OK,0,0.055002,15.969055,-9.198307,-24.364787,1.840595,0.023679,0.969551,2631.627282,0.001075,0.0,0.0
2,100ECDICEN_PGR276 B1_FULL MIX_3,100ECDICEN_PGR276 B1_FULL MIX.wav,10,15,OK,0,0.053938,16.342974,-8.990311,-24.623177,2.123389,0.025163,0.818542,3237.939448,0.001836,0.0,0.0
3,100ECDICEN_PGR276 B1_FULL MIX_4,100ECDICEN_PGR276 B1_FULL MIX.wav,15,20,OK,0,0.050941,16.819484,-9.025300,-25.115421,2.788394,0.028129,0.905393,2821.844361,0.001203,0.0,0.0
4,100ECDICEN_PGR276 B1_FULL MIX_5,100ECDICEN_PGR276 B1_FULL MIX.wav,20,25,OK,0,0.042641,18.379378,-8.989745,-26.324977,4.638039,0.049317,0.949445,3066.137187,0.001990,0.0,0.0


Celda 6 — Sanity check rápido

In [9]:
# Ver si hay NaN graves
nan_report = features_df.isna().mean().sort_values(ascending=False)
nan_report.head(15)

clip_id                   0.0
file_name                 0.0
start_time_s              0.0
end_time_s                0.0
qc_result                 0.0
y_bin                     0.0
rms_mean                  0.0
crest_factor_db           0.0
true_peak_dbfs            0.0
short_term_lufs_mean      0.0
loudness_var_proxy_db     0.0
silence_ratio             0.0
stereo_correlation        0.0
spectral_centroid_mean    0.0
spectral_flatness_mean    0.0
dtype: float64